# Baseline CNN

Execução do pipeline experimental reutilizável da CNN baseline no PlantVillage. O treinamento e todas as análises desta etapa usam somente os conjuntos de treino e validação. **O conjunto de teste permanece reservado para a avaliação final e não é avaliado neste notebook.**

## Caminhos e ambiente

A célula abaixo localiza o projeto localmente. No Google Colab, ela monta o Drive e usa `MyDrive/plant-disease-classification` por padrão, clonando automaticamente o repositório quando necessário. Assim, dados e resultados persistem após o encerramento do runtime. Para usar apenas o armazenamento temporário de `/content`, defina `MOUNT_DRIVE = False`. Todos os artefatos da execução ficam em `results/baseline_cnn/`.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPOSITORY_URL = "https://github.com/murilodc/plant-disease-classification.git"
COLAB_PROJECT_DIR = Path("/content/plant-disease-classification")
LEGACY_COLAB_PROJECT_DIR = Path("/content/tcc-plant-disease-classification")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/plant-disease-classification")
LEGACY_DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/TCC")
AUTO_CLONE_REPOSITORY = True
MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")


def is_project_root(path: Path) -> bool:
    return (path / "src" / "plantvillage_pytorch.py").is_file()


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    if IN_COLAB:
        candidates.extend(
            [
                DRIVE_PROJECT_DIR,
                LEGACY_DRIVE_PROJECT_DIR,
                COLAB_PROJECT_DIR,
                LEGACY_COLAB_PROJECT_DIR,
            ]
        )

    for candidate in candidates:
        if is_project_root(candidate):
            return candidate

    if IN_COLAB and AUTO_CLONE_REPOSITORY:
        clone_target = DRIVE_PROJECT_DIR if MOUNT_DRIVE else COLAB_PROJECT_DIR
        if clone_target.exists():
            raise FileNotFoundError(
                f"{clone_target} já existe, mas não contém o projeto completo. "
                "Remova ou renomeie esse diretório e execute a célula novamente."
            )
        print("Clonando o repositório para o runtime do Colab...")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(clone_target)]
        )
        if is_project_root(clone_target):
            return clone_target

    raise FileNotFoundError(
        "Projeto não encontrado. Envie o repositório completo ao Colab, "
        "monte o Google Drive ou habilite AUTO_CLONE_REPOSITORY."
    )


BASE_DIR = find_project_root()
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
BASELINE_RESULTS_DIR = RESULTS_DIR / "baseline_cnn"
SRC_DIR = BASE_DIR / "src"

# Ajuste estes caminhos se os arquivos estiverem em outro local no Colab.
ZIP_PATH = DATA_DIR / "data.zip"
LEAF_MAP_PATH = DATA_DIR / "leaf_grouping" / "leaf-map.json"
RAW_METADATA_CSV = RESULTS_DIR / "plantvillage_metadata_raw_color.csv"
METADATA_CSV = RESULTS_DIR / "plantvillage_metadata_split.csv"
IMAGE_ROOT = Path("/content/plantvillage_color") if IN_COLAB else DATA_DIR / "plantvillage_color"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Diretório src não encontrado: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Base:", BASE_DIR)
print("Dados:", DATA_DIR)
print("Resultados da baseline:", BASELINE_RESULTS_DIR)
print("Imagens:", IMAGE_ROOT)

## Dependências

As dependências ausentes são instaladas automaticamente. O nome de importação `sklearn` corresponde ao pacote `scikit-learn`.

In [ ]:
required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}
missing_packages = [
    package
    for module_name, package in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display

## Configuração da execução

In [ ]:
SEED = 42
NUM_CLASSES = 38
BATCH_SIZE = 32
MAX_EPOCHS = 30
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-4
NUM_WORKERS = 2 if IN_COLAB else 0
REQUIRE_CUDA_IN_COLAB = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if IN_COLAB and REQUIRE_CUDA_IN_COLAB and device.type != "cuda":
    raise RuntimeError(
        "GPU não disponível. No Colab, selecione Ambiente de execução > "
        "Alterar tipo de ambiente de execução > T4 GPU e execute novamente."
    )
print("Dispositivo selecionado automaticamente:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from plantvillage_audit import build_metadata_dataframe, save_metadata_csv
from plantvillage_pytorch import extract_raw_color_from_zip
from plantvillage_split import save_split_metadata_csv, split_metadata_by_leaf_id
from train_baseline import train_baseline

## Download, metadados e extração do PlantVillage

Esta etapa reaproveita as rotinas existentes no projeto. Downloads, geração do CSV de splits e extração só acontecem quando os respectivos arquivos ainda não existem.

In [ ]:
DOWNLOAD_DATA_IF_MISSING = True
EXTRACT_IMAGES_IF_MISSING = True

if DOWNLOAD_DATA_IF_MISSING and (not ZIP_PATH.exists() or not LEAF_MAP_PATH.exists()):
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import hf_hub_download

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        ZIP_PATH = Path(
            hf_hub_download(
                repo_id="mohanty/PlantVillage",
                filename="data.zip",
                repo_type="dataset",
                local_dir=str(DATA_DIR),
            )
        )
    if not LEAF_MAP_PATH.exists():
        LEAF_MAP_PATH = Path(
            hf_hub_download(
                repo_id="mohanty/PlantVillage",
                filename="leaf_grouping/leaf-map.json",
                repo_type="dataset",
                local_dir=str(DATA_DIR),
            )
        )

if not METADATA_CSV.exists():
    if not ZIP_PATH.exists() or not LEAF_MAP_PATH.exists():
        raise FileNotFoundError(
            "CSV de split ausente e arquivos base não encontrados. "
            "Ajuste ZIP_PATH e LEAF_MAP_PATH para os caminhos corretos."
        )
    print("CSV de split ausente; gerando com a lógica atual do repositório.")
    metadata = build_metadata_dataframe(ZIP_PATH, LEAF_MAP_PATH)
    save_metadata_csv(metadata, RAW_METADATA_CSV)
    metadata_split = split_metadata_by_leaf_id(metadata)
    save_split_metadata_csv(metadata_split, METADATA_CSV)
else:
    print("CSV de split encontrado:", METADATA_CSV)

if EXTRACT_IMAGES_IF_MISSING:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"ZIP não encontrado para extração: {ZIP_PATH}")
    print("Verificando/completando imagens raw/color em:", IMAGE_ROOT)
    extraction_summary = extract_raw_color_from_zip(ZIP_PATH, IMAGE_ROOT)
    print(extraction_summary)
    if int(extraction_summary['total']) != 54_305:
        raise RuntimeError("Extração incompleta: esperadas 54.305 imagens raw/color.")
elif not IMAGE_ROOT.is_dir() or not any(IMAGE_ROOT.iterdir()):
    raise FileNotFoundError(f"Diretório de imagens vazio ou ausente: {IMAGE_ROOT}")

## Conferência dos splits

A tabela é apenas uma conferência dos metadados. O split `test` não é usado no treinamento, na seleção do checkpoint, na cronometragem de inferência nem no cálculo das métricas abaixo.

In [ ]:
metadata_df = pd.read_csv(METADATA_CSV)
required_columns = {'zip_path', 'classe', 'leaf_id', 'split'}
missing_columns = sorted(required_columns - set(metadata_df.columns))
if missing_columns:
    raise KeyError(f"Colunas ausentes no CSV de split: {missing_columns}")
expected_split_sizes = {'train': 38_008, 'validation': 8_172, 'test': 8_125}
actual_split_sizes = metadata_df['split'].value_counts().to_dict()
if actual_split_sizes != expected_split_sizes:
    raise ValueError(f"Tamanhos de split inesperados: {actual_split_sizes}")
if metadata_df['classe'].nunique() != 38:
    raise ValueError("O CSV de split deve conter exatamente 38 classes.")
if metadata_df.groupby('leaf_id')['split'].nunique().max() != 1:
    raise ValueError("Data leakage: há leaf_id presente em mais de um split.")

split_sizes = (
    metadata_df["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="num_images")
)
display(split_sizes)
print("Avaliação durante o desenvolvimento: validation")
print("Avaliação final reservada: test (não utilizado neste notebook)")

## Treinamento e avaliação da baseline

`train_baseline` concentra a criação dos DataLoaders e do modelo, o resumo programático da arquitetura, o treinamento cronometrado, a restauração do melhor checkpoint e a avaliação completa na validação. Em CUDA, a função também faz sincronização para as medições e warm-up antes da inferência cronometrada.

In [ ]:
model, history, checkpoint_path = train_baseline(
    metadata_csv=METADATA_CSV,
    image_root=IMAGE_ROOT,
    output_dir=BASELINE_RESULTS_DIR,
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
    num_classes=NUM_CLASSES,
    num_workers=NUM_WORKERS,
    seed=SEED,
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=MIN_DELTA,
)

print("Melhor checkpoint:", checkpoint_path)

## Arquitetura registrada

O resumo salvo contém parâmetros totais e treináveis, convoluções, pooling, dropout, classificador final e número de classes de saída.

In [ ]:
architecture_path = BASELINE_RESULTS_DIR / "architecture_summary.json"
with architecture_path.open(encoding="utf-8") as architecture_file:
    architecture = json.load(architecture_file)

print(model)
display(
    pd.Series(
        {
            "model_class": architecture["model_class"],
            "total_parameters": architecture["total_parameters"],
            "trainable_parameters": architecture["trainable_parameters"],
            "num_convolutional_layers": architecture["num_convolutional_layers"],
            "num_output_classes": architecture["num_output_classes"],
        },
        name="architecture",
    ).to_frame()
)
display(pd.DataFrame(architecture["convolutional_layers"]))
display(pd.DataFrame(architecture["pooling_layers"]))
display(pd.DataFrame(architecture["dropout_layers"]))
display(pd.DataFrame([architecture["classification_layer"]]))

## Histórico e tempos

O histórico inclui somente as épocas executadas. O total de treinamento soma os intervalos de treino e validação das épocas. A inferência é medida de ponta a ponta sobre o DataLoader de validação, incluindo carregamento, transferência, forward e coleta das predições.

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

timing_path = BASELINE_RESULTS_DIR / "timing_summary.json"
with timing_path.open(encoding="utf-8") as timing_file:
    timing_summary = json.load(timing_file)

training_timing = timing_summary["training"]
inference_timing = timing_summary["validation_inference"]
print(f"Épocas executadas: {training_timing['epochs_executed']} / {MAX_EPOCHS}")
print(f"Early stopping: {'sim' if training_timing['early_stopping'] else 'não'}")
print(f"Melhor época: {training_timing['best_epoch']}")
print(f"Melhor validation loss: {training_timing['best_validation_loss']:.6f}")
print(f"Tempo total de treinamento: {training_timing['total_time_seconds']:.2f} s")
print(f"Tempo médio por época: {training_timing['average_time_per_epoch_seconds']:.2f} s")
print(f"Tempo total de inferência (validation): {inference_timing['total_time_seconds']:.4f} s")
print(
    "Tempo médio por imagem (validation): "
    f"{inference_timing['average_time_per_image_seconds']:.8f} s"
)
print(
    "Throughput (validation): "
    f"{inference_timing['throughput_images_per_second']:.2f} imagens/s"
)
print(f"Warm-up CUDA executado: {inference_timing['warmup_batches']} batch(es)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["validation_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Época", ylabel="CrossEntropyLoss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["validation_accuracy"], label="validation")
axes[1].set(title="Accuracy", xlabel="Época", ylabel="Accuracy")
axes[1].legend()

axes[2].bar(history_df["epoch"], history_df["epoch_time_seconds"])
axes[2].set(title="Tempo por época", xlabel="Época", ylabel="Segundos")

for axis in axes:
    axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Métricas de validação

As métricas globais incluem accuracy e agregações macro e weighted. A tabela por classe contém precision, recall, F1-score e suporte para cada uma das 38 classes.

In [ ]:
validation_metrics_path = BASELINE_RESULTS_DIR / "validation_metrics.csv"
class_metrics_path = BASELINE_RESULTS_DIR / "validation_metrics_by_class.csv"

validation_metrics_df = pd.read_csv(validation_metrics_path)
class_metrics_df = pd.read_csv(class_metrics_path)

display(validation_metrics_df.T.rename(columns={0: "value"}))
display(class_metrics_df)

## Matriz de confusão da validação

In [ ]:
confusion_matrix_path = BASELINE_RESULTS_DIR / "validation_confusion_matrix.png"
display(Image(filename=str(confusion_matrix_path), width=1100))

## Artefatos gerados

In [ ]:
artifacts = sorted(path for path in BASELINE_RESULTS_DIR.iterdir() if path.is_file())
artifacts_df = pd.DataFrame(
    {
        "arquivo": [path.name for path in artifacts],
        "caminho": [str(path) for path in artifacts],
        "tamanho_bytes": [path.stat().st_size for path in artifacts],
    }
)
display(artifacts_df)

metadata_path = BASELINE_RESULTS_DIR / "experiment_metadata.json"
with metadata_path.open(encoding="utf-8") as metadata_file:
    experiment_metadata = json.load(metadata_file)

print("\nBaseline CNN concluída")
print(f"Épocas executadas: {training_timing['epochs_executed']} / {MAX_EPOCHS}")
print(f"Early stopping: {'sim' if training_timing['early_stopping'] else 'não'}")
print(f"Melhor época: {training_timing['best_epoch']}")
print(f"Melhor validation loss: {training_timing['best_validation_loss']:.6f}")
print("Validation accuracy do melhor modelo: " f"{experiment_metadata['best_model_validation_accuracy']:.6f}")
print(f"Tempo total de treinamento: {training_timing['total_time_seconds']:.2f} s")
print(f"Tempo médio por época: {training_timing['average_time_per_epoch_seconds']:.2f} s")
print("Tempo médio de inferência por imagem: " f"{inference_timing['average_time_per_image_seconds']:.8f} s")
print(f"Throughput: {inference_timing['throughput_images_per_second']:.2f} imagens/s")
print(f"Parâmetros treináveis: {architecture['trainable_parameters']}")
print("Todos os resultados da baseline estão em:", BASELINE_RESULTS_DIR)